# Session 3 — From Notebook to Trading System

This session turns the exact Session 2 inference contract into an auditable
historical paper-trading replay. It proves batch/stream parity, records every
event in SQLite, exercises operational guardrails, and reconciles the final
state.

**90-minute allocation:** artifact validation (10), event timing and parity
(20), SQLite replay (25), guardrails and failure injection (20), reconciliation
and production gap (15).


## 1. Reconnect to the Exact Run

Session 3 never reconstructs an architecture from memory and never substitutes
random weights. The named Drive bundle must contain a checksummed Session 2
checkpoint with its feature order, scaler, topology, weights, and threshold.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/algocolab')
    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                'git', 'clone', '--depth', '1',
                'https://github.com/yhilpisch/algocolab.git',
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    RUNS_ROOT = Path('/content/drive/MyDrive/algo/runs')
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'data' / 'eod_data.csv').is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    RUNS_ROOT = Path(
        '/Users/yves/Google Drive/My Drive/algo/runs'
    )
REFERENCE_RUN_ID = 'session1-reference-20260907-v2'
RUN_ID = os.environ.get('WEBINAR_RUN_ID', REFERENCE_RUN_ID)
PERSIST_RESULTS = IN_COLAB and RUN_ID != REFERENCE_RUN_ID
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Run: {RUN_ID}')


In [ ]:
from src.artifacts import RunBundle
from src.models import load_model_checkpoint
from src.session3 import persist_session_three, run_session_three
bundle = RunBundle.open(
    RUNS_ROOT,
    RUN_ID,
    required_session=2,
)
model, contract = load_model_checkpoint(
    bundle.path / 'session_2/model.pt'
)
print(contract['model_config'])
print(contract['feature_names'])
print(f"Threshold: {contract['threshold']:.2f}")


## 2. Event Timing Before Infrastructure

For target bar `t`, the replay builds features only from prices through
`t-1`, sets the position at the start of the bar, realizes return `t`, deducts
cost for absolute position turnover, and then evaluates drawdown. Transport
through ZeroMQ or another message bus does not change this timing contract.


The wealth update uses log returns and one-way turnover:

\[
\Delta p_t=|p_t-p_{t-1}|,\qquad
r_t^{\mathrm{net}}=p_t r_t-c\Delta p_t,\qquad
V_t=V_{t-1}\exp(r_t^{\mathrm{net}}).
\]

With \(c=0.5\) basis points, a direct reversal has two turnover units and
costs 1 basis point.


## 3. Run the Historical Paper-Trading Replay

SQLite is the durable event ledger for the demonstration. It records ticks,
signals, orders, and portfolio states under the same run ID. This remains a
paper-trading simulation—not a broker-connected production service.


In [ ]:
temporary = TemporaryDirectory()
database_path = Path(temporary.name) / 'paper_trading.db'
results = run_session_three(
    bundle,
    PROJECT_ROOT / 'data' / 'eod_data.csv',
    database_path,
    max_drawdown=0.10,
)
results.reconciliation


## 4. Prove Batch/Stream Parity

The deployment path rebuilds each feature vector from the historical buffer,
one observation at a time. Those features, probabilities, and positions must
match the batch research path within floating-point tolerance.


In [ ]:
results.parity


In [ ]:
assert results.reconciliation['parity_passed']
assert results.reconciliation['bars_recorded'] == 499
print('Parity release gate: PASS')


## 5. Inspect the Ledger and Risk Response

The 10% drawdown guardrail overrides the model, creates an explicit flattening
order, sets the system to `HALTED`, and prevents re-entry. State continues to be
recorded after the halt so the audit trail remains complete.


In [ ]:
results.orders.tail()


In [ ]:
import matplotlib.pyplot as plt
telemetry = results.telemetry.copy()
telemetry['timestamp'] = telemetry['timestamp'].astype('datetime64[ns]')
axes = telemetry.plot(
    x='timestamp',
    y=['nav'],
    figsize=(10, 4),
    color=['#002D5A'],
    legend=False,
)
axes.set(title='Paper-trading NAV with risk halt', ylabel='NAV')
axes.grid(alpha=0.2)
plt.show()


## 6. Inject Failures Deliberately

A guardrail is credible only when an adverse input produces observable,
testable evidence. The suite injects a nine-day stale-data gap, an 18.18% price
jump, and a loss large enough to require a flattening order.


In [ ]:
results.failure_tests


In [ ]:
assert results.failure_tests['passed'].all()
assert results.reconciliation['final_position'] == 0
assert results.reconciliation['risk_flatten_orders'] == 1
print('Operational checks: PASS')


## 7. Reconcile and Persist

The reconciliation summarizes row counts, final position and NAV, maximum
drawdown, turnover, halt state, and parity. A participant run adds the SQLite
database and portable CSV/JSON evidence to the same Drive bundle.


In [ ]:
if PERSIST_RESULTS:
    code_commit = subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    persist_session_three(
        bundle,
        results,
        code_commit=code_commit,
    )
    print(f'Session 3 persisted: {bundle.path}')
else:
    print('Reference/local replay: Drive persistence disabled.')


> **Production deepening beyond the live skeleton**
>
> - broker authentication, acknowledgements, rejects, and idempotent order IDs;
> - heartbeat supervision, latency budgets, reconnect and replay semantics;
> - database/broker reconciliation and restart recovery;
> - secrets management, container hardening, monitoring, and alert escalation;
> - slippage, partial fills, financing, exposure limits, and kill switches;
> - model/data drift policy, approval workflow, and rollback.
>
> Passing this replay validates the educational contract. It is not production
> certification and does not authorize live trading.


## Series Takeaway

The three sessions form one traceable chain: test a narrow predictability
hypothesis, evaluate a non-linear alternative without contaminating the test
set, and deploy the exact frozen decision rule into an auditable paper-trading
replay. Negative economic results remain part of the evidence.
